In [41]:
import os
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

if "DASHSCOPE_API_KEY" in os.environ and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = os.environ["DASHSCOPE_API_KEY"]

# 1. 准备 Document
documents = [
    Document(
        page_content="试用期员工不享受带薪年假。",
        metadata={
            "source": "员工手册2026",
            "page": 23,
            "status": "active",
            "year": 2026,
        },
    ),
    Document(
        page_content="正式员工每年享有10天带薪年假。",
        metadata={
            "source": "员工手册2026",
            "page": 23,
            "status": "active",
            "year": 2026,
        },
    ),
    Document(
        page_content="单笔报销超过5000元，需要部门负责人审批。",
        metadata={
            "source": "财务制度2026",
            "page": 12,
            "status": "active",
            "year": 2026,
        },
    ),
    Document(
    page_content="试用期员工享有3天带薪年假。",
    metadata={
        "source": "员工手册2024",
        "page": 20,
        "status": "archived",
        "year": 2024,
    },
    )
]



# 2. 初始化 Embedding
embeddings = OpenAIEmbeddings(
    model="qwen3.7-text-embedding",  # 1. 必须写成关键字参数 model=
    openai_api_base="https://llm-v4mqkweosjvc0she.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
    check_embedding_ctx_length=False,  # 2. 阿里云 DashScope 兼容接口必须加上此项
)

# 3. 初始化 InMemoryVectorStore
vector_store = InMemoryVectorStore(
    embedding=embeddings
)

# 4. add_documents
vector_store.add_documents(documents)

# 5. similarity_search
query = "我还没转正，能休假吗？"
#query = "公司有没有火星移民补贴？"
# 过滤出高于阈值的有效文档

SCORE_THRESHOLD = 0.7

def retrieve_company_knowledge(query: str) -> dict:
    results = vector_store.similarity_search_with_score(
        query,
        k=3,
        filter=lambda doc:
            doc.metadata.get("status") == "active"
            and doc.metadata.get("year") == 2026
    )

    valid_results = [
        (doc, score)
        for doc, score in results
        if score >= SCORE_THRESHOLD
    ]

    if not valid_results:
        return {
            "status": "NO_RELEVANT_KNOWLEDGE",
            "results": []
        }

    return {
        "status": "OK",
        "results": [
            {
                "content": doc.page_content,
                "score": score,
                "source": doc.metadata.get("source"),
                "page": doc.metadata.get("page"),
                "year": doc.metadata.get("year"),
            }
            for doc, score in valid_results
        ]
    }

retrieve_company_knowledge(query)

        

# for i in range(3):
#     results = vector_store.similarity_search_with_score(
#         query,
#         k= i+1,
#         filter=lambda doc: doc.metadata.get("status") == "active" and doc.metadata.get("year") == 2026
#     )

#     SCORE_THRESHOLD = 0.7
    
#     valid_results = [
#     (doc, score)
#     for doc, score in results
#     if score >= SCORE_THRESHOLD
#     ]
    
#     if not valid_results:
#         print("未在知识库中检索到足够相关的规定。")
#     else:
#         for doc, score in valid_results:
#             print("k=",i+1)  
#             for doc, score in valid_results:
#                 print("内容：", doc.page_content)
#                 print(score)
#                 print("来源：", doc.metadata)
#                 print("-----")
        


{'status': 'OK',
 'results': [{'content': '试用期员工不享受带薪年假。',
   'score': 0.7822699882083324,
   'source': '员工手册2026',
   'page': 23,
   'year': 2026}]}